In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import uuid


In [0]:
%sql
use catalog novacart_workspace;
create schema if not exists bronze_schema ;




In [0]:
%sql
create table if not exists novacart_workspace.bronze_schema.ingestion_control(
    layer string,
    table_name string,
    ts_col string,
    pk_col string,
    last_successful_ts timestamp,
    last_successful_pk bigint,
    last_run_id string,
    rows_wriiten bigint,
    run_status string,
    updated_at timestamp
)
using delta

In [0]:
table_config={
    "orders":{"pk_col":"order_id","ts_col":"updated_at"},
    "products":{"pk_col":"product_id","ts_col":"updated_at"},
    "payments":{"pk_col":"payment_id","ts_col":"processed_at"}
}

bronze_run_id=str(uuid.uuid4())
print(bronze_run_id)

In [0]:
def get_last_successful_watermark(table_name):
    ctrl = (
        spark.read.table(
            "novacart_workspace.bronze_schema.ingestion_control"
        )
        .filter(
            (col("layer") == "bronze") &
            (col("table_name") == table_name) &
            (col("run_status") == "success")
        )
        .orderBy(col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None, None
    else:
        return (
            rows[0]["last_successful_ts"],
            rows[0]["last_successful_pk"]
        )



In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_wriiten,run_id):
    control_df=spark.createDataFrame([("bronze",table_name,ts_col,pk_col,last_ts,
                                       int(last_pk) if last_pk is not None else None, 
                                       run_id,int(rows_wriiten),"success",datetime.utcnow())],
                                     schema="""layer string, table_name string,ts_col string,pk_col string,last_successful_ts timestamp,last_successful_pk bigint, last_run_id string, rows_wriiten bigint, run_status string, updated_at timestamp """)
    
    df = DeltaTable.forName(spark,"novacart_workspace.bronze_schema.ingestion_control")
    (df.alias("t").merge(control_df.alias("s"),
                         "t.table_name = s.table_name and t.layer = s.layer")
     .whenMatchedUpdate(set={"ts_col":"s.ts_col","pk_col":"s.pk_col","last_successful_ts":"s.last_successful_ts","last_successful_pk":"s.last_successful_pk","last_run_id":"s.last_run_id","rows_wriiten":"s.rows_wriiten","run_status":"s.run_status","updated_at":"s.updated_at" })
     .whenNotMatchedInsertAll().execute())
     

In [0]:
for table_name,cfg in table_config.items():
    pk_col=cfg["pk_col"]
    ts_col=cfg['ts_col']
    source_table=f"novacast_sql_server_connection_catalog.dbo.{table_name}"
    target_table=f"novacart_workspace.bronze_schema.{table_name}_raw"
    last_successful_ts,last_successful_pk=get_last_successful_watermark(table_name)
    print(f"\n=== processing{table_name} ==")
    print(f"last_successful_ts: {last_successful_ts}")
    print(f"last_successful_pk: {last_successful_pk}")

    source_df=spark.read.table(source_table).withColumn(ts_col,col(ts_col).cast("timestamp"))
    

    if last_successful_ts is None:
        rows_to_load=source_df
    else:
        condition = (
    (col(ts_col) > lit(last_successful_ts)) |
    (
        (col(ts_col) == lit(last_successful_ts)) &
        (col(pk_col).cast("long") > lit(last_successful_pk))
    )
)
        rows_to_load=source_df.filter(condition)


    rows_to_load=rows_to_load.withColumn("bronze_ingested_at",current_timestamp()).withColumn("bronze_run_id",lit(bronze_run_id)).withColumn("bronze_source_table",lit(source_table))

    rows_count=rows_to_load.count()
    print(f"{table_name}rows to load: {rows_count}")

    if rows_count ==0:
        print(f"no new data to load for {table_name}")
        upsert_bronze_control(table_name,ts_col,pk_col,last_successful_ts,last_successful_pk,rows_count,bronze_run_id)
        continue
    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    max_ts=rows_to_load.agg(max(ts_col).alias("max_ts")).collect()[0]["max_ts"]
    max_pk=(rows_to_load.filter(col(ts_col)==lit(max_ts)).agg(max(pk_col).alias("max_pk")).collect()[0]["max_pk"])
    
    upsert_bronze_control(table_name,ts_col,pk_col,max_ts,max_pk,rows_count,run_id=bronze_run_id)
    print(f"wrote {rows_count} rows to {target_table}")
   

